In [1]:

import torch, sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("Enable GPU in Runtime settings before continuing.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64


In [2]:

!pip -q install ultralytics huggingface_hub

from huggingface_hub import hf_hub_download
from pathlib import Path

model_path = hf_hub_download(
    repo_id="AurevinP/PULSE_AI_Models",
    filename="ball_tracker_200_epoch_92L_25M.pt"
)

print("Downloaded model:", model_path)
print("File size (MB):", round(Path(model_path).stat().st_size / 1e6, 2))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 206.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 kB 9.1 MB/s eta 0:00:00


ball_tracker_200_epoch_92L_25M.pt: reconstructing file:   0%|          |  0.00B / 52.1MB            

ball_tracker_200_epoch_92L_25M.pt: downloading bytes:           |  0.00B            

Downloaded model: /root/.cache/huggingface/hub/models--AurevinP--PULSE_AI_Models/snapshots/5c7b65a062557736baa036af15ca469f8a5b2847/ball_tracker_200_epoch_92L_25M.pt
File size (MB): 52.06


In [3]:

from ultralytics import YOLO
import torch

model = YOLO(model_path)

print("Model loaded successfully")
print("Model classes:", model.names)
print("GPU available:", torch.cuda.is_available())


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Model loaded successfully
Model classes: {0: 'ball'}
GPU available: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

zip_files = list(drive_root.glob("*.zip"))

print("ZIP files found:", len(zip_files))
for i, f in enumerate(zip_files, 1):
    print(f"{i}. {f.name} — {f.stat().st_size / 1e9:.2f} GB")


ZIP files found: 0


In [6]:

from pathlib import Path

athletiq_drive = Path("/content/drive/MyDrive/AthletiQ")

print("AthletiQ folder exists:", athletiq_drive.exists())

for f in athletiq_drive.rglob("*.zip"):
    print(f"{f}  |  {f.stat().st_size / 1e9:.2f} GB")


AthletiQ folder exists: True
/content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip  |  18.31 GB


In [7]:

import shutil
from pathlib import Path

total, used, free = shutil.disk_usage("/content")

print(f"Colab free space: {free / 1e9:.2f} GB")

zip_source = Path(
    "/content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip"
)

assert zip_source.is_file(), "Dataset ZIP not found!"

print(f"ZIP size: {zip_source.stat().st_size / 1e9:.2f} GB")

if free / 1e9 < 40:
    print("WARNING: Check storage before copying and extracting.")
else:
    print("Enough initial space for the copy; extraction space must also be checked.")


Colab free space: 75.35 GB
ZIP size: 18.31 GB
Enough initial space for the copy; extraction space must also be checked.


In [8]:

import shutil
from pathlib import Path

source = Path("/content/drive/MyDrive/AthletiQ/soccernet_ball_yolo.zip")
destination = Path("/content/soccernet_ball_yolo.zip")

shutil.copy2(source, destination)

print("Copy completed:", destination)
print(f"Copied size: {destination.stat().st_size / 1e9:.2f} GB")


Copy completed: /content/soccernet_ball_yolo.zip
Copied size: 18.31 GB


In [9]:

import zipfile
from pathlib import Path

zip_path = Path("/content/soccernet_ball_yolo.zip")
extract_dir = Path("/content/soccernet_ball_yolo")

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files in archive:", len(z.infolist()))
    print("Uncompressed size (GB):",
          round(sum(i.file_size for i in z.infolist()) / 1e9, 2))

    bad_file = z.testzip()
    if bad_file:
        raise RuntimeError(f"Corrupted archive member: {bad_file}")

    z.extractall(extract_dir)

print("Extraction completed:", extract_dir)


Files in archive: 159011
Uncompressed size (GB): 18.3
Extraction completed: /content/soccernet_ball_yolo


In [10]:

from pathlib import Path

root = Path("/content/soccernet_ball_yolo")

for p in root.rglob("*.yaml"):
    print("\nYAML:", p)
    print(p.read_text()[:2500])



YAML: /content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml
path: "/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/processed/soccernet_ball_yolo"
train: images/train
val: images/val
test: images/test

names:
  0: ball



In [11]:

from pathlib import Path
import yaml

yaml_path = Path(
    "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"
)

dataset_root = yaml_path.parent

config = yaml.safe_load(yaml_path.read_text())
config["path"] = str(dataset_root)

yaml_path.write_text(yaml.safe_dump(config, sort_keys=False))

print(yaml_path.read_text())


path: /content/soccernet_ball_yolo/soccernet_ball_yolo
train: images/train
val: images/val
test: images/test
names:
  0: ball



In [12]:

for split in ["train", "val", "test"]:
    image_dir = dataset_root / "images" / split
    label_dir = dataset_root / "labels" / split

    print(f"\n{split}:")
    print("Image directory exists:", image_dir.exists())
    print("Label directory exists:", label_dir.exists())

    if image_dir.exists():
        images = [
            p for p in image_dir.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        ]
        print("Images in split:", len(images))



train:
Image directory exists: True
Label directory exists: True
Images in split: 55500

val:
Image directory exists: True
Label directory exists: True
Images in split: 11250

test:
Image directory exists: True
Label directory exists: True
Images in split: 12750


In [13]:

from ultralytics import YOLO
import torch

# Reuse model_path from the successful P.U.L.S.E download.
model = YOLO(model_path)

results = model.val(
    data=str(yaml_path),
    imgsz=640,  # Change to your exact Stage 1 validation size
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,
    verbose=True
)

print("\n--- P.U.L.S.E RESULTS ---")
print(f"Precision:  {results.box.mp:.4f}")
print(f"Recall:     {results.box.mr:.4f}")
print(f"mAP@50:     {results.box.map50:.4f}")
print(f"mAP@50-95:  {results.box.map:.4f}")


Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2005.9±849.5 MB/s, size: 257.0 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/val... 11250 images, 986 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11250/11250 1.3Kit/s 8.4s
val: New cache created: /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1407/1407 5.7it/s 4:07
                   all      11250      10636      0.376      0.217      0.143     0.0339
Speed: 0.6ms preprocess, 18.1ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val

--- P.U.L.S.E RESULTS ---
Precision:  0.3758
Recall:     0.2165
mAP@50:     0.1431
mAP@50-95:  0.0339


In [14]:

from pathlib import Path

root = Path("/content/drive/MyDrive/AthletiQ")

for p in root.rglob("*.pt"):
    print(f"{p}  |  {p.stat().st_size / 1e6:.1f} MB")


/content/drive/MyDrive/AthletiQ/models/best.pt  |  10.8 MB
/content/drive/MyDrive/AthletiQ/models/v9b_best.pt  |  22.7 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/epoch0.pt  |  21.3 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/epoch1.pt  |  21.3 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/epoch2.pt  |  21.3 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/epoch3.pt  |  21.3 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/epoch4.pt  |  21.3 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/last.pt  |  5.5 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/best.pt  |  5.5 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt  |  5.5 MB
/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/last.pt  |  5.5 MB


In [15]:

from pathlib import Path
from ultralytics import YOLO
import csv
import torch

DATA_YAML = "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"

candidates = {
    "models_best": "/content/drive/MyDrive/AthletiQ/models/best.pt",
    "models_v9b_best": "/content/drive/MyDrive/AthletiQ/models/v9b_best.pt",
    "stage1_5ep_best": "/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_5ep/weights/best.pt",
    "stage2_best": "/content/drive/MyDrive/AthletiQ/ball_detector_yolo11n_stage2/weights/best.pt",
}

results_list = []

for name, path in candidates.items():
    if not Path(path).is_file():
        print(f"SKIPPING: {name} — file not found")
        continue

    print(f"\n{'='*50}\nEvaluating: {name}\n{path}")

    model = YOLO(path)
    metrics = model.val(
        data=DATA_YAML,
        imgsz=640,
        batch=1,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=0,
        max_det=50,
        split="val",
        plots=False,
        verbose=False,
    )

    row = {
        "model": name,
        "checkpoint": path,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    }
    results_list.append(row)

    print(
        f"Precision={row['precision']:.4f} | "
        f"Recall={row['recall']:.4f} | "
        f"mAP50={row['mAP50']:.4f} | "
        f"mAP50-95={row['mAP50_95']:.4f}"
    )

if results_list:
    results_list.sort(key=lambda r: r["mAP50_95"], reverse=True)

    output = "/content/athletiq_checkpoint_comparison.csv"
    with open(output, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results_list[0].keys())
        writer.writeheader()
        writer.writerows(results_list)

    print("\n--- RANKING BY mAP50-95 ---")
    for row in results_list:
        print(row["model"], round(row["mAP50"], 4),
              round(row["mAP50_95"], 4))

    print("\nCSV saved:", output)



Evaluating: models_best
/content/drive/MyDrive/AthletiQ/models/best.pt
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2552.1±873.0 MB/s, size: 191.5 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/val.cache... 11250 images, 986 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11250/11250 2.0Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11250/11250 42.1it/s 4:27
                   all      11250      10636      0.478      0.212      0.163     0.0458
Speed: 0.4ms preprocess, 8.8ms inference, 0.0ms loss, 1.0ms postprocess per image
Precision=0.4783 | Recall=0.2115 | mAP50=0.1633 | mAP50-95=0.0458

Evaluating: models_v9b_best
/content/drive/MyDrive/AthletiQ/models/v9b_best.pt
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 

In [16]:

from pathlib import Path
from ultralytics import YOLO
import torch
import yaml

model_path = globals().get("model_path")

if not model_path or not Path(model_path).is_file():
    raise FileNotFoundError(
        "P.U.L.S.E checkpoint not found. Set model_path "
        "to the downloaded .pt checkpoint."
    )

data_yaml = Path(
    "/content/soccernet_ball_yolo/"
    "soccernet_ball_yolo/data.yaml"
)

assert data_yaml.is_file(), f"Dataset YAML missing: {data_yaml}"

config = yaml.safe_load(data_yaml.read_text())
dataset_root = Path(config["path"])

print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")
print("Dataset root:", dataset_root)
print("Classes:", config["names"])

for split in ["train", "val"]:
    images = dataset_root / "images" / split
    labels = dataset_root / "labels" / split
    assert images.is_dir(), f"Missing {images}"
    assert labels.is_dir(), f"Missing {labels}"
    print(split, "images:", len(list(images.iterdir())))
    print(split, "labels:", len(list(labels.iterdir())))

model = YOLO(model_path)
print("P.U.L.S.E classes:", model.names)
print("Checkpoint verified.")


GPU: Tesla T4
Dataset root: /content/soccernet_ball_yolo/soccernet_ball_yolo
Classes: {0: 'ball'}
train images: 55500
train labels: 55500
val images: 11250
val labels: 11250
P.U.L.S.E classes: {0: 'ball'}
Checkpoint verified.


In [20]:

from ultralytics import YOLO
import torch

model = YOLO(model_path)

results = model.train(
    data=str(data_yaml),
    epochs=8,
    imgsz=640,
    batch=36,
    device=0,
    workers=2,
    project="/content/athletiq_pulse",
    name="soccernet_finetune_4ep",
    exist_ok=True,
    pretrained=True,
    save=True,
    plots=True,
    seed=42,
)

print("Training completed.")
print("Best checkpoint:",
      "/content/athletiq_pulse/soccernet_finetune_4ep/weights/best.pt")


Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=36, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/.cache/huggingface/hub/models--AurevinP--PULSE_AI_Models/snapshots/5c7b65a062557736baa03

In [21]:

from pathlib import Path
import shutil

source = Path(
    "/content/athletiq_pulse/"
    "soccernet_finetune_4ep/weights/best.pt"
)

destination = Path(
    "/content/drive/MyDrive/AthletiQ/models/"
    "pulse_soccernet_finetuned_8ep_best.pt"
)

assert source.is_file(), f"Checkpoint missing: {source}"
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

print("Saved:", destination)
print("Size (MB):", round(destination.stat().st_size / 1e6, 2))


Saved: /content/drive/MyDrive/AthletiQ/models/pulse_soccernet_finetuned_8ep_best.pt
Size (MB): 52.0


In [24]:

from ultralytics import YOLO
from pathlib import Path
import torch

data = "/content/soccernet_ball_yolo/soccernet_ball_yolo/data.yaml"

models = {
    "PULSE_finetuned": str(destination),
    "YOLO11n_stage1_5ep": (
        "/content/drive/MyDrive/AthletiQ/"
        "ball_detector_yolo11n_5ep/weights/best.pt"
    ),
    "YOLO11n_stage2": (
        "/content/drive/MyDrive/AthletiQ/"
        "ball_detector_yolo11n_stage2/weights/best.pt"
    ),
}

for name, path in models.items():
    print(f"\n{'='*50}\n{name}")

    model = YOLO(path)
    metrics = model.val(
        data=data,
        imgsz=1280,
        batch=36,
        device=0,
        workers=0,
        max_det=50,
        split="val",
        plots=True,
        verbose=False,
    )

    print(f"Precision:  {metrics.box.mp:.4f}")
    print(f"Recall:     {metrics.box.mr:.4f}")
    print(f"mAP@50:     {metrics.box.map50:.4f}")
    print(f"mAP@50–95:  {metrics.box.map:.4f}")



PULSE_finetuned
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1700.3±1137.4 MB/s, size: 237.1 KB)
val: Scanning /content/soccernet_ball_yolo/soccernet_ball_yolo/labels/val.cache... 11250 images, 986 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11250/11250 3.1Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 3.5s/it 18:07
                   all      11250      10636      0.654      0.636      0.583      0.234
Speed: 0.9ms preprocess, 73.9ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /content/runs/detect/val-8
Precision:  0.6544
Recall:     0.6358
mAP@50:     0.5825
mAP@50–95:  0.2345

YOLO11n_stage1_5ep
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers,